In [ ]:

import math
import os
import random
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["Arial", "SimHei"]  # 支持中英文
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="ticks")


def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


seed_everything(42)

def augment_with_shuffled_negatives(
    df: pd.DataFrame, ratio: float = 0.5, seed: int = 42
) -> pd.DataFrame:
    if ratio <= 0:
        return df.copy()

    pos_df = df[df["label"] == 1].copy()
    neg_df = df[df["label"] == 0].copy()
    if len(pos_df) == 0 or len(neg_df) == 0:
        return df.copy()

    num_to_add = int(len(pos_df) * ratio)
    rng = np.random.default_rng(seed)

    sampled_pos_apts = rng.choice(pos_df["aptamer"].values, size=num_to_add, replace=True)
    sampled_neg_prots = rng.choice(neg_df["protein"].values, size=num_to_add, replace=True)

    aug_df = pd.DataFrame(
        {"label": 0, "aptamer": sampled_pos_apts, "protein": sampled_neg_prots}
    )

    merged = pd.concat([df, aug_df], ignore_index=True)
    return merged.sample(frac=1.0, random_state=seed).reset_index(drop=True)

DNA_HYDROPATHY = {"A": 1.8, "C": 2.5, "G": -0.4, "T": 0.0}


def get_dna_features(seq):
    seq = seq.upper()
    length = len(seq) if len(seq) > 0 else 1
    gc_content = sum(1 for base in seq if base in "GC") / length
    purine_content = sum(1 for base in seq if base in "AG") / length
    hydro = np.mean([DNA_HYDROPATHY.get(base, 0.0) for base in seq])
    return [gc_content, purine_content, hydro, length / 100.0]


PROT_CHARGE = {"R": 1.0, "K": 1.0, "H": 0.5, "D": -1.0, "E": -1.0}
PROT_HYDROPATHY = {
    "A": 1.8,
    "R": -4.5,
    "N": -3.5,
    "D": -3.5,
    "C": 2.5,
    "Q": -3.5,
    "E": -3.5,
    "G": -0.4,
    "H": -3.2,
    "I": 4.5,
    "L": 3.8,
    "K": -3.9,
    "M": 1.9,
    "F": 2.8,
    "P": -1.6,
    "S": -0.8,
    "T": -0.7,
    "W": -0.9,
    "Y": -1.3,
    "V": 4.2,
}


def get_prot_features(seq):
    seq = seq.upper()
    length = len(seq) if len(seq) > 0 else 1
    charge = sum(PROT_CHARGE.get(aa, 0.0) for aa in seq) / length
    hydro = np.mean([PROT_HYDROPATHY.get(aa, 0.0) for aa in seq])
    aromatic = sum(1 for aa in seq if aa in "FYW") / length
    isoelectric_approx = sum(1 for aa in seq if aa in "RKH") / (
        sum(1 for aa in seq if aa in "DE") + 1e-5
    )
    return [
        charge,
        hydro,
        aromatic,
        min(isoelectric_approx, 5.0),
        length / 1000.0,
    ]

NUC_VOCAB = {"<PAD>": 0, "A": 1, "C": 2, "G": 3, "T": 4, "<UNK>": 5}
AA_LIST = [
    "<PAD>",
    "A",
    "R",
    "N",
    "D",
    "C",
    "Q",
    "E",
    "G",
    "H",
    "I",
    "L",
    "K",
    "M",
    "F",
    "P",
    "S",
    "T",
    "W",
    "Y",
    "V",
    "<UNK>",
]
AA_VOCAB = {aa: i for i, aa in enumerate(AA_LIST)}


class AptamerProteinDataset(Dataset):
    def __init__(self, df, max_apt_len=60, max_prot_len=500):
        self.df = df.reset_index(drop=True)
        self.max_apt_len = max_apt_len
        self.max_prot_len = max_prot_len

    def __len__(self):
        return len(self.df)

    def _encode_seq(self, seq, vocab, max_len):
        tokens = [
            vocab.get(char, vocab.get("<UNK>")) for char in str(seq).upper()
        ]
        if len(tokens) < max_len:
            tokens = tokens + [vocab["<PAD>"]] * (max_len - len(tokens))
        else:
            tokens = tokens[:max_len]
        return tokens

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        apt_seq = str(row["aptamer"])
        prot_seq = str(row["protein"])
        label = float(row["label"])

        apt_tokens = self._encode_seq(apt_seq, NUC_VOCAB, self.max_apt_len)
        prot_tokens = self._encode_seq(prot_seq, AA_VOCAB, self.max_prot_len)

        apt_bio = get_dna_features(apt_seq)
        prot_bio = get_prot_features(prot_seq)

        return {
            "apt_seq": torch.tensor(apt_tokens, dtype=torch.long),
            "prot_seq": torch.tensor(prot_tokens, dtype=torch.long),
            "apt_bio": torch.tensor(apt_bio, dtype=torch.float32),
            "prot_bio": torch.tensor(prot_bio, dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.float32),
        }

class MultiScaleDilatedConv(nn.Module):
    def __init__(self, in_dim=128, out_dim=128):
        super().__init__()
        # 3 个不同膨胀率的空洞卷积分支，获取不同尺度的感受野
        self.conv_d1 = nn.Conv1d(in_dim, out_dim // 4, kernel_size=3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(in_dim, out_dim // 4, kernel_size=3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(in_dim, out_dim // 2, kernel_size=3, padding=4, dilation=4)
        
        self.bn = nn.BatchNorm1d(out_dim)
        self.act = nn.GELU()

    def forward(self, x):
        # x: [B, C, L]
        out1 = self.conv_d1(x)
        out2 = self.conv_d2(x)
        out4 = self.conv_d4(x)
        out = torch.cat([out1, out2, out4], dim=1) # [B, 128, L]
        return self.act(self.bn(out + x))        


class CrossGatedAttention(nn.Module):
    def __init__(self, d_model=128, nhead=4, dropout=0.2):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True
        )
        self.gate_proj = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value):
        attn_out, _ = self.mha(query=query, key=key_value, value=key_value)
        gate = torch.sigmoid(self.gate_proj(query))
        gated_attn = attn_out * gate
        
        out = self.norm(query + self.dropout(gated_attn))
        return out


class BioPhysicalModulator(nn.Module):
    def __init__(self, bio_dim=9, feat_dim=128):
        super().__init__()
        self.bio_mlp = nn.Sequential(
            nn.Linear(bio_dim, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.scale_proj = nn.Linear(32, feat_dim)
        self.shift_proj = nn.Linear(32, feat_dim)

    def forward(self, seq_feat, bio_input):
        # seq_feat: [B, 128]
        # bio_input: [B, 9]
        bio_emb = self.bio_mlp(bio_input)      # [B, 32]
        scale = self.scale_proj(bio_emb)       # [B, 128]
        shift = self.shift_proj(bio_emb)       # [B, 128]
        
        # FiLM: Output = Scale * Feature + Shift
        modulated_feat = seq_feat * (1.0 + scale) + shift
        return modulated_feat, bio_emb


# ==============================================================================
#GCAMNet
# ==============================================================================
class GCAMNet(nn.Module):
    def __init__(self, len_nuc_vocab=10, len_aa_vocab=25):
        super().__init__()
        # --- A. Apt：Embedding + Self-Attention  ---
        self.apt_embed = nn.Embedding(len_nuc_vocab, 64, padding_idx=0)
        self.apt_proj = nn.Linear(64, 128)
        self.apt_self_attn = nn.MultiheadAttention(embed_dim=128, num_heads=4, dropout=0.1, batch_first=True)
        self.apt_norm = nn.LayerNorm(128)
        self.apt_pool = nn.AdaptiveAvgPool1d(1)

        # --- B. prot：Embedding + BiGRU + Dilated ResBlock (捕捉空间口袋) ---
        self.prot_embed = nn.Embedding(len_aa_vocab, 64, padding_idx=0)
        self.prot_gru = nn.GRU(64, 64, num_layers=1, batch_first=True, bidirectional=True) # [B, L, 128]
        self.prot_dilated_block = MultiScaleDilatedConv(in_dim=128, out_dim=128)

        # --- C. attention ---
        self.cross_gated_attn = CrossGatedAttention(d_model=128, nhead=4, dropout=0.2)

        # --- D. FiLM ---
        self.bio_modulator = BioPhysicalModulator(bio_dim=4 + 5, feat_dim=128)

        # 128 (Aptamer Global) + 128 (Modulated Cross Attn) + 32 (Bio Embedding) = 288
        total_dim = 128 + 128 + 32
        self.classifier = nn.Sequential(
            nn.Linear(total_dim, 96),
            nn.BatchNorm1d(96),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(96, 1)
        )

    def forward(self, apt_seq, prot_seq, apt_bio, prot_bio):

        x_apt_emb = self.apt_proj(self.apt_embed(apt_seq))             # [B, L_apt, 128]
        apt_self_out, _ = self.apt_self_attn(x_apt_emb, x_apt_emb, x_apt_emb)
        apt_seq_feat = self.apt_norm(x_apt_emb + apt_self_out)          # [B, L_apt, 128]
        apt_global = torch.mean(apt_seq_feat, dim=1)                    # [B, 128]


        x_prot_emb = self.prot_embed(prot_seq)                          # [B, L_prot, 64]
        gru_out, _ = self.prot_gru(x_prot_emb)                          # [B, L_prot, 128]
        prot_seq_feat = self.prot_dilated_block(gru_out.transpose(1, 2)).transpose(1, 2) # [B, L_prot, 128]

        attn_out = self.cross_gated_attn(query=apt_seq_feat, key_value=prot_seq_feat) # [B, L_apt, 128]
        
        attn_mean = torch.mean(attn_out, dim=1)
        attn_max = torch.max(attn_out, dim=1)[0]
        attn_global = 0.7 * attn_mean + 0.3 * attn_max                   # [B, 128]

        bio_input = torch.cat([apt_bio, prot_bio], dim=1)
        modulated_attn, bio_feat = self.bio_modulator(attn_global, bio_input)

        fusion = torch.cat([apt_global, modulated_attn, bio_feat], dim=1) # [B, 288]
        logits = self.classifier(fusion).squeeze(-1)
        
        probs = torch.sigmoid(logits)                                    
        return probs


class FBCELoss(nn.Module):
    def __init__(self, alpha=0.35, gamma=1.5):  # 1.5
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, probs, targets):
        eps = 1e-7
        probs = torch.clamp(probs, eps, 1.0 - eps)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        focal_weight = alpha_t * ((1 - pt) ** self.gamma)
        bce = -(
            targets * torch.log(probs) + (1 - targets) * torch.log(1 - probs)
        )
        return torch.mean(focal_weight * bce)


class EMA:
    def __init__(self, model, decay=0.98):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (
                    1.0 - self.decay
                ) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name])


# ==============================================================================
# 6. train
# ==============================================================================
def train_one_epoch(model, dataloader, optimizer, criterion, ema, device):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        apt_seqs = batch["apt_seq"].to(device)
        prot_seqs = batch["prot_seq"].to(device)
        apt_bio = batch["apt_bio"].to(device)
        prot_bio = batch["prot_bio"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        probs = model(apt_seqs, prot_seqs, apt_bio, prot_bio)
        loss = criterion(probs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        ema.update()
        total_loss += loss.item() * len(labels)
    return total_loss / len(dataloader.dataset)


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    all_probs = []
    all_targets = []
    for batch in dataloader:
        apt_seqs = batch["apt_seq"].to(device)
        prot_seqs = batch["prot_seq"].to(device)
        apt_bio = batch["apt_bio"].to(device)
        prot_bio = batch["prot_bio"].to(device)
        labels = batch["label"].to(device)

        probs = model(apt_seqs, prot_seqs, apt_bio, prot_bio)
        all_probs.extend(probs.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())
    return np.array(all_probs), np.array(all_targets)



def compute_metrics(all_targets, all_probs):
    auc_score = roc_auc_score(all_targets, all_probs)
    precision_vals, recall_vals, _ = precision_recall_curve(
        all_targets, all_probs
    )
    aupr_score = auc(recall_vals, precision_vals)


    best_j = -1.0 
    best_thresh = 0.5

    for thresh in np.arange(0.1, 0.9, 0.02):
        preds = (all_probs >= thresh).astype(int)

        cm_tmp = confusion_matrix(all_targets, preds)
        tn, fp, fn, tp = cm_tmp.ravel()

        sens = tp / (tp + fn + 1e-5)
        spec = tn / (tn + fp + 1e-5)

        youden_j = sens + spec - 1

        if youden_j > best_j:
            best_j = youden_j
            best_thresh = thresh

    final_preds = (all_probs >= best_thresh).astype(int)
    cm = confusion_matrix(all_targets, final_preds)
    tn, fp, fn, tp = cm.ravel()


    sensitivity = tp / (tp + fn + 1e-5)
    specificity = tn / (tn + fp + 1e-5)
    accuracy = accuracy_score(all_targets, final_preds)
    final_f1 = f1_score(all_targets, final_preds, zero_division=0)

    return {
        "auc": auc_score,
        "aupr": aupr_score,
        "f1": final_f1,  
        "best_thresh": best_thresh,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "accuracy": accuracy,
    }


def plot_roc_curve(roc_data_list):
    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=300)

    mean_fpr = np.linspace(0, 1, 200)
    tprs = []
    auc_list = []
    fold_colors = ["#7F8C8D", "#95A5A6", "#BDC3C7", "#90A4AE", "#78909C"]

    for idx, (fpr, tpr, fold_auc) in enumerate(roc_data_list):
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        auc_list.append(fold_auc)

        ax.plot(
            fpr,
            tpr,
            lw=1.3,
            color=fold_colors[idx % len(fold_colors)],
            alpha=0.6,
            linestyle="--",
            label=f"Fold {idx+1} (AUC = {fold_auc:.4f})",
        )

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = np.mean(auc_list)
    std_auc = np.std(auc_list)

    main_color = "#C0392B"
    ax.plot(
        mean_fpr,
        mean_tpr,
        color=main_color,
        lw=2.8,
        label=f"Mean ROC (AUC = {mean_auc:.4f} $\pm$ {std_auc:.4f})",
    )

    std_tpr = np.std(tprs, axis=0)
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    ax.fill_between(
        mean_fpr,
        tprs_lower,
        tprs_upper,
        color=main_color,
        alpha=0.15,
        edgecolor=None,
    )

    ax.plot([0, 1], [0, 1], linestyle=":", lw=1.5, color="#555555", alpha=0.7)

    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=12, labelpad=8)
    ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=12, labelpad=8)
    ax.set_title("ROC Curves", fontsize=13.5, weight="bold", pad=12)

    ax.legend(loc="lower right", fontsize=9.5, frameon=True, facecolor="white", framealpha=0.9, edgecolor="#E0E0E0")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5, color="#CCCCCC")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig("5_fold_roc_curve.jpg", dpi=300, bbox_inches="tight")
    plt.show()

def plot_pr_curve(pr_data_list):
    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=300)

    mean_recall = np.linspace(0, 1, 200)
    precisions = []
    aupr_list = []

    fold_colors = ["#7F8C8D", "#95A5A6", "#BDC3C7", "#90A4AE", "#78909C"]

    for idx, (recall, precision, fold_aupr) in enumerate(pr_data_list):
        idx_sort = np.argsort(recall)
        r_sorted = recall[idx_sort]
        p_sorted = precision[idx_sort]

        interp_precision = np.interp(mean_recall, r_sorted, p_sorted)
        precisions.append(interp_precision)
        aupr_list.append(fold_aupr)

        ax.plot(
            recall,
            precision,
            lw=1.3,
            color=fold_colors[idx % len(fold_colors)],
            alpha=0.6,
            linestyle="--",
            label=f"Fold {idx+1} (AUPR = {fold_aupr:.4f})",
        )

    mean_precision = np.mean(precisions, axis=0)
    mean_aupr = np.mean(aupr_list)
    std_aupr = np.std(aupr_list)

    main_color = "#16A085"
    ax.plot(
        mean_recall,
        mean_precision,
        color=main_color,
        lw=2.8,
        label=f"Mean PR (AUPR = {mean_aupr:.4f} $\pm$ {std_aupr:.4f})",
    )

    std_precision = np.std(precisions, axis=0)
    precisions_upper = np.minimum(mean_precision + std_precision, 1)
    precisions_lower = np.maximum(mean_precision - std_precision, 0)
    ax.fill_between(
        mean_recall,
        precisions_lower,
        precisions_upper,
        color=main_color,
        alpha=0.15,
        edgecolor=None,
    )

    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel("Recall (Sensitivity)", fontsize=12, labelpad=8)
    ax.set_ylabel("Precision", fontsize=12, labelpad=8)
    ax.set_title("Precision-Recall Curves", fontsize=13.5, weight="bold", pad=12)

    ax.legend(loc="lower left", fontsize=9.5, frameon=True, facecolor="white", framealpha=0.9, edgecolor="#E0E0E0")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5, color="#CCCCCC")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig("5_fold_aupr_curve.jpg", dpi=300, bbox_inches="tight")
    plt.show()


def plot_best_fold_confusion_matrix(best_fold_idx, targets, probs, threshold):
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(targets, preds)

    cm_ratio = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    labels = np.array(
        [
            [f"{v}\n({r:.1%})" for v, r in zip(row_val, row_ratio)]
            for row_val, row_ratio in zip(cm, cm_ratio)
        ]
    )

    fig, ax = plt.subplots(figsize=(5.2, 4.5), dpi=300)

    sns.heatmap(
        cm,
        annot=labels,
        fmt="",
        cmap="YlOrBr",
        cbar=True,
        square=True,
        linewidths=1.5,
        linecolor="white",  
        annot_kws={"fontsize": 11.5, "weight": "bold", "color": "#2C3E50"},
        xticklabels=["Negative (0)", "Positive (1)"],
        yticklabels=["Negative (0)", "Positive (1)"],
        ax=ax,
        cbar_kws={"shrink": 0.82},
    )

    ax.set_xlabel("Predicted Label", fontsize=11, labelpad=8, weight="bold")
    ax.set_ylabel("True Label", fontsize=11, labelpad=8, weight="bold")
    ax.set_title(
        "Confusion Matrix",
        fontsize=12,
        weight="bold",
        pad=12,
    )

    plt.tight_layout()
    plt.savefig("confusion_matrix.jpg", dpi=300, bbox_inches="tight")
    plt.show()

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🌟 cuda: {device}")

    csv_path = "Data26_merged_all.csv"
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No {csv_path}")

    df = pd.read_csv(csv_path)
    print(f"📊 read dataset: {csv_path} | Number: {len(df)} ")

    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_metrics_list = []
    roc_data_list = []
    pr_data_list = []

    best_global_auc = -1.0
    best_fold_idx = -1
    best_fold_targets = None
    best_fold_probs = None
    best_fold_thresh = 0.5

    epochs = 50

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label"]), 1):
        print(f"\n" + "-" * 40)
        print(f"🌀 training Fold {fold}/{n_splits}")
        print("-" * 40)

        train_df = df.iloc[train_idx].copy()
        val_df = df.iloc[val_idx].copy()

        train_df = augment_with_shuffled_negatives(
            train_df, ratio=0.5, seed=42 + fold
        )

        train_dataset = AptamerProteinDataset(train_df)
        val_dataset = AptamerProteinDataset(val_df)

        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

        model = GCAMNet().to(device)
        criterion = FBCELoss(alpha=0.35, gamma=1.5)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=1e-3, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, eta_min=1e-5
        )
        ema = EMA(model, decay=0.98)

        best_fold_auc = 0.0
        saved_probs = None
        saved_targets = None

        for epoch in range(1, epochs + 1):
            train_loss = train_one_epoch(
                model, train_loader, optimizer, criterion, ema, device
            )
            scheduler.step()

            ema.apply_shadow()
            probs, targets = evaluate(model, val_loader, device)
            ema.restore()

            metrics = compute_metrics(targets, probs)

            if metrics["auc"] > best_fold_auc:
                best_fold_auc = metrics["auc"]
                saved_probs = probs
                saved_targets = targets

                ema.apply_shadow()
                torch.save(
                    model.state_dict(), f"best_ms_lgfn_v3_fold{fold}.pth"
                )
                ema.restore()

            if epoch % 10 == 0 or epoch == epochs:
                print(
                    f"Fold {fold} | Epoch {epoch:02d}/{epochs:02d} | Loss: {train_loss:.4f} | "
                    f"Val AUC: {metrics['auc']:.4f} | Best Val AUC: {best_fold_auc:.4f}"
                )


        fold_final_metrics = compute_metrics(saved_targets, saved_probs)
        fold_metrics_list.append(fold_final_metrics)

        fpr, tpr, _ = roc_curve(saved_targets, saved_probs)
        precision, recall, _ = precision_recall_curve(saved_targets, saved_probs)

        roc_data_list.append((fpr, tpr, fold_final_metrics["auc"]))
        pr_data_list.append((recall, precision, fold_final_metrics["aupr"]))

        print(f"✅ Fold {fold} best AUC: {fold_final_metrics['auc']:.4f}")


        if fold_final_metrics["auc"] > best_global_auc:
            best_global_auc = fold_final_metrics["auc"]
            best_fold_idx = fold
            best_fold_targets = saved_targets
            best_fold_probs = saved_probs
            best_fold_thresh = fold_final_metrics["best_thresh"]


    print("\n" + "=" * 50)
    print("🏆 5-Fold Cross Validation ")
    print("=" * 50)


    metrics_to_calc = [
        "auc",
        "aupr",
        "sensitivity",
        "specificity",
        "f1",
        "accuracy",
    ]
    fold_stats = {}

    for m_key in metrics_to_calc:
        values = [m[m_key] for m in fold_metrics_list]
        fold_stats[m_key] = {"mean": np.mean(values), "std": np.std(values)}


    for m_key in metrics_to_calc:
        fold_vals = [round(m[m_key], 4) for m in fold_metrics_list]
        mean_val = fold_stats[m_key]["mean"]
        std_val = fold_stats[m_key]["std"]

        metric_name = m_key.upper().ljust(11)

        print(f"▸ {metric_name} 各 Fold: {fold_vals}")
        print(
            f"  └─► Mean ± STD: {mean_val:.4f} ± {std_val:.4f}\n"
        )

    print("=" * 70)

    print(" 🌟 5-Fold  (Mean $\pm$ Std) :")
    print(
        f"  • Mean AUC         : {fold_stats['auc']['mean']:.4f} $\pm$ {fold_stats['auc']['std']:.4f}"
    )
    print(
        f"  • Mean AUPR        : {fold_stats['aupr']['mean']:.4f} $\pm$ {fold_stats['aupr']['std']:.4f}"
    )
    print(
        f"  • Mean Sensitivity : {fold_stats['sensitivity']['mean']:.4f} $\pm$ {fold_stats['sensitivity']['std']:.4f}"
    )
    print(
        f"  • Mean Specificity : {fold_stats['specificity']['mean']:.4f} $\pm$ {fold_stats['specificity']['std']:.4f}"
    )
    print(
        f"  • Mean F1 Score    : {fold_stats['f1']['mean']:.4f} $\pm$ {fold_stats['f1']['std']:.4f}"
    )
    print(
        f"  • Mean Accuracy    : {fold_stats['accuracy']['mean']:.4f} $\pm$ {fold_stats['accuracy']['std']:.4f}"
    )
    print("-" * 50)
    print(f"🔥 NO. {best_fold_idx} best (AUC = {best_global_auc:.4f})")

   

    plot_roc_curve(roc_data_list)
    plot_pr_curve(pr_data_list)


    plot_best_fold_confusion_matrix(
        best_fold_idx, best_fold_targets, best_fold_probs, best_fold_thresh
    )



if __name__ == "__main__":
    main()
